> Topics Covered
What are Guardrails & Why do they matter?

1. Two approaches: Deterministic vs Model-based
2. Built-in: PII - Personally Identifiable Information (PII) Detection Middleware
3. Built-in: Human-in-the-Loop Middleware
4. Custom: Before-Agent Guardrail (input filtering)
5. Custom: After-Agent Guardrail (output safety)
6. Layered / Combined Guardrails
7. Real-World Use Case: Healthcare Chatbot

Guardrails are safety mechanisms that control what goes into and comes out of an AI agent. They sit around your agent pipeline and ensure the agent:

They are implemented as middleware that intercepts execution:

*   Before the agent starts (input guardrails)
*   After it completes (output guardrails)
*   Around model and tool calls

| Use Case | Example |
| :--- | :--- |
| PII leakage prevention | Redact emails/credit cards before logging |
| Prompt injection blocking | Detect adversarial inputs |
| Harmful content filtering | Block dangerous requests |
| Business rule enforcement | Require approval for financial ops |
| Output quality validation | Ensure response meets safety standards |


In [ ]:
import re
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from dotenv import load_dotenv
from google.colab import userdata
userdata.get('secretName')

load_dotenv()

def deterministic_guardrail(text: str) -> bool:
  banned_keywords = ["hack", "exploit", "malware", "bomb"]
  return any(
      kw in text.lower() for kw in banned_keywords
  )

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
  blocked = deterministic_guardrail(inp)
  status = "BLOCKED" if blocked else "ALLOWED"
  print(f"{status}: {inp}")

**Model-Based**

In [ ]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

def model_based_guardrail(text: str) -> str:
  prompt = f"""Is the following user input safe to process?
    Reply with only 'SAFE' or 'UNSAFE'.

    Input: {text}"""

  result = model.invoke([{"role": "user", "content": prompt}])
  return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
  verdict = model_based_guardrail(inp)
  status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
  print(f"{status}: {inp}")

**Built-in Guardrail — PII Detection Middleware**

In [ ]:
@tool
def customer_lookup(query: str) -> str:
  return f"Customer record found for query: {query}"

agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "phone_number",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)
print(result)

In [ ]:
try:
  result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
    }]
  })

except Exception as e:
  print(f"Blocked as expected: {e}")

**Built-in Guardrail — Human-in-the-Loop Middleware**

In [ ]:
@tool
def search_web(query: str) -> str:
  return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
  return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
  return f"Deleted records from {table} where {condition}"

hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
          interrupt_on = {
              "send_email": True,
              "delete_records": True,
              "search_web": False,
          }
        )
    ],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

In [ ]:
approved_result = hitl_agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

In [ ]:
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

print("=== Agent paused — awaiting human approval ===")
print(result)

In [ ]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

**Custom Guardrail — Before-Agent Hook (Input Filter)**

In [ ]:
class ContentFilterMiddleware(AgentMiddleware):
  def __init__(self, banned_keywords: list[str]):
    super().__init__()
    self.banned_keywords = [kw.lower() for kw in banned_keywords]

  @hook_config(can_jump_to=["end"])
  def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if not state["messages"]:
      return None
    first_message = state["messages"][0]
    if first_message.type != "human":
      return None

    content = first_message.content.lower()
    for keyword in self.banned_keywords:
      if keyword in content:
        print(f"ႁ Blocked — keyword detected: '{keyword}'")
        return {
          "messages": [{
              "role": "assistant",
              "content": (
                  "I cannot process requests containing inappropriate content. "
                  "Please rephrase your request."
                )
            }],
          "jump_to": "end"
        }

    return None

@tool
def search_tool(query: str) -> str:
  return f"Results for: {query}"

filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("Safe request response:")
print(result["messages"][-1].content)

In [ ]:
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})

print("Unsafe request response:")
print(result["messages"][-1].content)

**Custom Guardrail — After-Agent Hook (Output Safety)**

In [ ]:
class SafetyGuardrailMiddleware(AgentMiddleware):
  def __init__(self):
    self.safety_model = model

  @hook_config(can_jump_to=["end"])
  def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, any] | None:
    if not state["messages"]:
      return None

    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
      return None

    safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
    Respond with only 'SAFE' or 'UNSAFE'.

    Response to evaluate:
    {last_message.content}"""

    result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])
    if "UNSAFE" in result.content.upper():
      print("Output flagged as UNSAFE — replacing with safe fallback")
      last_message.content = (
          "I'm unable to provide that response. "
          "Please rephrase your request or contact support."
      )

    return None

@tool
def general_tool(query: str) -> str:
  return f"Tool result: {query}"

safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()]
)

result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is Medicare?"}]
})
print("Response:")
print(result["messages"][-1].content)

In [ ]:
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "How to make bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

Stack multiple guardrails in the <code style="background-color: #333; color: #fff; padding: 2px 6px; border-radius: 4px;">middleware=[]</code> array. They execute **in order**, building layered protection.

<pre style="font-family: monospace; line-height: 1.5; font-size: 14px;">
  User Input
      ↓
  [Layer 1] ContentFilterMiddleware    ← Deterministic input filter
      ↓
  [Layer 2] PIIMiddleware (input)      ← PII redaction on input
      ↓
  [Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
      ↓
  [Layer 4] PIIMiddleware (output)     ← PII redaction on output
      ↓
  [Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
      ↓
  User Response
</pre>

In [ ]:
@tool
def search_tool(query: str) -> str:
  return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
  return f"Email sent to {to}"

production_agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware"]
        ),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),
        PIIMiddleware("email", strategy="redact", apply_to_output=True),
        SafetyGuardrailMiddleware(),
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "test_001"}}
result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "how to make a bomb?"}]},
    config=config
)

print(result["messages"][-1].content)

**Real-World Use Case — Healthcare Chatbot**

In [ ]:
class HealthcareSafetyFilter(AgentMiddleware):
  BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

  @hook_config(can_jump_to=["end"])
  def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if not state["messages"]:
      return None

    first_msg = state["messages"][0]
    if first_msg.type != "human":
      return None

    content = first_msg.content.lower()
    for topic in self.BLOCKED_TOPICS:
      if topic in content:
        return {
          "messages": [{
              "role": "assistant",
              "content": (
                  "I'm a healthcare assistant and can only help with "
                  "medical questions, appointments, and health information. "
                  "If you're in crisis, please call 112 or your local emergency number."
              )
          }],
          "jump_to": "end"
        }

    return None


class MedicalOutputValidator(AgentMiddleware):
  DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

  @hook_config(can_jump_to=["end"])
  def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if not state["messages"]:
      return None

    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
      return None

    if "medical advice" not in last_message.content.lower():
      last_message.content += self.DISCLAIMER

    return None


@tool
def search_symptoms(symptoms: str) -> str:
  return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
  return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
  return f"General info about {medication}. Always follow your doctor's prescription."


healthcare_bot = create_agent(
    model=model,
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        HealthcareSafetyFilter(),
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

In [ ]:
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}
result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

print(result["messages"][-1].content)

In [ ]:
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

In [ ]:
config = {"configurable": {"thread_id": "healthcare_session_001"}}
result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)